# ДЗ 3. NumPy на данных «Дала Маркет»
Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

## Чтение кода — без запуска
1. `5 (2, 3)` — элемент во 2-й строке и 3-м столбце равен 5; форма массива 2×3.
2. `1 8` — `argmax` возвращает индекс первого максимума (индекс 1), `max` — само значение 8.
3. `99` — срез `a[::3]` это view, а не копия, поэтому `b[0] = 99` меняет `a[0]`.
4. `2 0.5` — условие `a > 3` верно для 5 и 7: `sum` = 2, `mean` = 2/4 = 0.5.
5. `(2, 3) 0.25` — `linspace(0, 1, 5)` = 0, 0.25, 0.5, 0.75, 1; индекс 1 даёт 0.25.

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "").replace("\xa0", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
is_test = (cust == "TEST")

print("Заказов:", amount.shape[0], "| dtype суммы:", amount.dtype, "| dtype paid:", paid.dtype)
print("Заказов клиента TEST:", is_test.sum())

## [База]

In [ ]:
# 1. Самый дорогой заказ
i = amount.argmax()
print(f"1. Позиция {i}: {amount[i]:,.0f} ₸, клиент {cust[i]}, город {city[i]}")
if cust[i] == "TEST":
    print("   Менеджеру: самый дорогой «заказ» принадлежит тестовому клиенту TEST — это не реальная продажа.")
    j = np.where(~is_test, amount, -np.inf).argmax()
    print(f"   Самый дорогой реальный заказ: позиция {j}, {amount[j]:,.0f} ₸, клиент {cust[j]}, город {city[j]}.")
else:
    print(f"   Менеджеру: самый крупный заказ на {amount[i]:,.0f} ₸ сделал клиент {cust[i]} из города {city[i]}.")

# 2. Оплаченные и неоплаченные
rev_paid, rev_unpaid = amount[paid].sum(), amount[~paid].sum()
print(f"\n2. Оплачено: {rev_paid:,.0f} ₸ | не оплачено: {rev_unpaid:,.0f} ₸")
print(f"   Менеджеру: в кассе {rev_paid:,.0f} ₸, ещё {rev_unpaid:,.0f} ₸ висят в неоплаченных заказах.")

# 3. Шымкент
m = (city == "Шымкент")
print(f"\n3. Шымкент: {m.sum()} заказов, доля {m.mean():.1%}")
print(f"   Менеджеру: Шымкент даёт {m.sum()} заказов — {m.mean():.1%} от всех.")

## [Стандарт]

In [ ]:
# 4. Скидка 15% на оплаченные заказы дороже 50 000 ₸
elig = paid & (amount > 50000)
loss_all = 0.15 * amount[elig].sum()
loss_clean = 0.15 * amount[elig & ~is_test].sum()
loss_test = 0.15 * amount[elig & is_test].sum()

print("4. Недополучим со всеми заказами:", f"{loss_all:,.0f} ₸")
print("   Недополучим без TEST:         ", f"{loss_clean:,.0f} ₸")
print("   Вклад TEST:                   ", f"{loss_test:,.0f} ₸")
print("   Суммы заказов TEST:", amount[is_test])
print("   Из них попало под скидку:", (elig & is_test).sum(), "из", is_test.sum())
if loss_clean > 0:
    print(f"   Разница в {loss_all/loss_clean:.1f} раза.")
print(f"   Менеджеру: реально недополучим около {loss_clean:,.0f} ₸; цифра {loss_all:,.0f} ₸ завышена из-за тестовых заказов TEST.")

In [ ]:
# 5. Доля доставок за 1–2 дня по городам (только заказы, где срок доставки известен)
res = {}
print("5. Город: доля 1–2 дня | заказов со сроком")
for c in np.unique(city):
    ok = (city == c) & ~np.isnan(days)
    n = ok.sum()
    if n:
        res[c] = ((days[ok] >= 1) & (days[ok] <= 2)).mean()
        print(f"   {c:<10} {res[c]:>6.1%} | {n}")
best, worst = max(res, key=res.get), min(res, key=res.get)
print(f"   Менеджеру: быстрее всего доставляют в г. {best} ({res[best]:.1%} за 1–2 дня), медленнее всего — в г. {worst} ({res[worst]:.1%}).")

In [ ]:
# 6. Средний чек: код AI считал amount.mean() по ВСЕМ заказам
bad = amount.mean()
good = amount[paid & ~is_test].mean()
print(f"6. Было (код AI): {round(bad):,} ₸ | стало: {round(good):,} ₸")
print("   Ошибка 1: в среднее попали неоплаченные заказы — денег с них нет.")
print("   Ошибка 2: в среднее попали тестовые заказы клиента TEST — это не покупатели.")
print(f"   Менеджеру: реальный средний чек — {round(good):,} ₸ (оплаченные заказы без TEST).")

## [Челлендж]

In [ ]:
# 7. Скорость: цикл for против маски
from time import perf_counter

t = perf_counter()
s_for = 0.0
for k in range(len(amount)):
    if paid[k]:
        s_for += amount[k]
t_for = perf_counter() - t

t = perf_counter()
for _ in range(1000):
    s_mask = amount[paid].sum()
t_mask = (perf_counter() - t) / 1000

print(f"7. Суммы совпали: {np.isclose(s_for, s_mask)}")
print(f"   Цикл: {t_for*1e6:.1f} мкс | маска: {t_mask*1e6:.1f} мкс | быстрее в {t_for/t_mask:.1f} раз")
print(f"   Менеджеру: маска быстрее цикла примерно в {t_for/t_mask:.0f} раз, на больших данных разница растёт.")

In [ ]:
# 8. A/B-тест
np.random.seed(42)
a = (np.random.rand(1000) < 0.05).sum()
b = (np.random.rand(1000) < 0.06).sum()
print(f"8. Один эксперимент: A = {a} покупок, B = {b} покупок")

b_win = tie = a_win = 0
for _ in range(100):
    a_ = (np.random.rand(1000) < 0.05).sum()
    b_ = (np.random.rand(1000) < 0.06).sum()
    if b_ > a_: b_win += 1
    elif b_ == a_: tie += 1
    else: a_win += 1
print(f"   За 100 повторов: B лучше A — {b_win}, ничья — {tie}, A лучше B — {a_win}")
print(f"   Менеджеру: хотя B на самом деле лучше, при 1000 клиентов в группе она выигрывает лишь в {b_win}% случаев, "
      f"а в {a_win + tie}% нет: разница в 1 п.п. тонет в случайном шуме. "
      "Чтобы надёжно её заметить, нужна выборка в разы больше.")